# LLM Long-Short Equity Strategy
## Powered by Fine-Tuned DeepSeek 8B + DefeatBeta API

This notebook implements a **quarterly long-short equity strategy** driven by a locally fine-tuned DeepSeek 8B language model. The model reads earnings call transcripts and news articles for S&P 500 stocks and outputs a forward-looking `llm_score` (0–100). Stocks are ranked cross-sectionally each quarter; the **top 10%** are longed and the **bottom 10%** are shorted with equal weighting.

**Research question:** Can a locally fine-tuned DeepSeek 8B model extract useful forward-looking signals from earnings calls and news, and can these signals generate positive long-short returns in a quarterly S&P 500 strategy?

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 0 — Configuration

In [2]:
# ============================================================
# Section 0: Configuration
# All user-editable parameters are here. Do not edit other cells.
# ============================================================

import os

# --- Model ---
BASE_MODEL_HF_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
LORA_ADAPTER_PATH = "/content/drive/MyDrive/CSE545/models/deepseek-financial-cot-lora/checkpoint-6300"

# --- Google Drive ---
DRIVE_BASE_PATH = "/content/drive/MyDrive/CSE545/LLM_Strategy"

# --- Universe ---
UNIVERSE_SIZE = None        # Set to None for full S&P 500
RANDOM_SEED = 42

# --- Time Horizon ---
NUM_QUARTERS = 12
REBALANCE_DELAY_DAYS = 45

# --- Portfolio ---
LONG_PCT = 0.10           # Top 10% long
SHORT_PCT = 0.10          # Bottom 10% short

# --- LLM Inference ---
MAX_INPUT_TOKENS = 3072
MAX_NEW_TOKENS = 300
LOAD_IN_4BIT = True
CHECKPOINT_EVERY = 50

print("[Config] ✅ Configuration loaded.")
print(f"  Base model       : {BASE_MODEL_HF_ID}")
print(f"  LoRA adapter     : {LORA_ADAPTER_PATH}")
print(f"  Drive base path  : {DRIVE_BASE_PATH}")
print(f"  Universe size    : {UNIVERSE_SIZE or 'Full S&P 500'}")
print(f"  Quarters         : {NUM_QUARTERS}")
print(f"  Long/Short pct   : {LONG_PCT}/{SHORT_PCT}")
print(f"  4-bit quant      : {LOAD_IN_4BIT}")

[Config] ✅ Configuration loaded.
  Base model       : deepseek-ai/DeepSeek-R1-Distill-Llama-8B
  LoRA adapter     : /content/drive/MyDrive/CSE545/models/deepseek-financial-cot-lora/checkpoint-6300
  Drive base path  : /content/drive/MyDrive/CSE545/LLM_Strategy
  Universe size    : Full S&P 500
  Quarters         : 12
  Long/Short pct   : 0.1/0.1
  4-bit quant      : True


## Section 1 — Install Dependencies

In [5]:
# # ============================================================
# # Section 1: Install Dependencies (run this cell, then restart runtime)
# # ============================================================
# print("[Dependencies] Installing packages...")

# Step 1: Install defeatbeta-api first (pulls numpy>=2.2, pandas>=3.0)
!pip install -q defeatbeta-api

# Step 2: Install ML/finance packages (compatible with new numpy/pandas)
!pip install -q \
    transformers>=4.40 peft>=0.10 bitsandbytes>=0.43 accelerate>=0.29 \
    sentencepiece protobuf matplotlib seaborn tqdm yfinance pyarrow \
    fastparquet

print("[Dependencies] ✅ All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 27.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.9/831.9 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 160.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 135.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 158.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Section 2 — Imports

In [4]:
# ============================================================
# Section 2: Imports
# ============================================================
print("[Imports] Loading libraries...")

import os
import re
import json
import math
import time
import warnings
import random
from datetime import datetime, timedelta, date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from tqdm.auto import tqdm

import requests
import yfinance as yf

from defeatbeta_api.data.ticker import Ticker

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print("[Imports] ✅ All libraries loaded.")

[Imports] Loading libraries...


ModuleNotFoundError: No module named 'defeatbeta_api'

## Section 3 — Mount Google Drive

In [ ]:
# ============================================================
# Section 3: Mount Google Drive
# ============================================================
print("[Drive] Mounting Google Drive...")

from google.colab import drive
drive.mount('/content/drive')

# Create directory structure
for subdir in ['data/raw/earnings_calls', 'data/raw/news', 'data/raw/prices',
               'data/processed', 'outputs']:
    os.makedirs(os.path.join(DRIVE_BASE_PATH, subdir), exist_ok=True)

print(f"[Drive] ✅ Drive mounted. Base path: {DRIVE_BASE_PATH}")
print(f"[Drive] Directory structure created.")

## Section 4 — Quarter Utilities

In [ ]:
# ============================================================
# Section 4: Quarter Utilities
# ============================================================
print("[Quarters] Computing calendar quarters...")

def get_quarter_info(year, quarter):
    """Return (quarter_label, start_date, end_date, rebalance_date) for a calendar quarter."""
    quarter_starts = {1: (1, 1), 2: (4, 1), 3: (7, 1), 4: (10, 1)}
    quarter_ends   = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}

    start = date(year, *quarter_starts[quarter])
    end   = date(year, *quarter_ends[quarter])
    rebalance = end + timedelta(days=REBALANCE_DELAY_DAYS)
    label = f"{year}Q{quarter}"
    return label, start, end, rebalance

def get_past_quarters(num_quarters):
    """Return the last `num_quarters` complete calendar quarters (rebalance date has passed)."""
    today = date.today()
    quarters = []
    # Go back far enough
    for year in range(today.year - 5, today.year + 1):
        for q in range(1, 5):
            label, start, end, rebalance = get_quarter_info(year, q)
            if rebalance < today:
                quarters.append({
                    'quarter': label,
                    'start_date': start,
                    'end_date': end,
                    'rebalance_date': rebalance
                })
    # Take the most recent num_quarters
    quarters = sorted(quarters, key=lambda x: x['start_date'])
    quarters = quarters[-num_quarters:]
    return quarters

QUARTERS = get_past_quarters(NUM_QUARTERS)

print(f"[Quarters] ✅ {len(QUARTERS)} complete quarters identified.")
print(f"  Earliest : {QUARTERS[0]['quarter']}  ({QUARTERS[0]['start_date']} to {QUARTERS[0]['end_date']})  rebalance: {QUARTERS[0]['rebalance_date']}")
print(f"  Latest   : {QUARTERS[-1]['quarter']}  ({QUARTERS[-1]['start_date']} to {QUARTERS[-1]['end_date']})  rebalance: {QUARTERS[-1]['rebalance_date']}")

## Section 5 — S&P 500 Universe

In [ ]:
# ============================================================
# Section 5: S&P 500 Universe
# ============================================================
print("[Universe] Fetching S&P 500 constituents...")

FALLBACK_TICKERS = [
    'AAPL', 'ABBV', 'ABT', 'ACN', 'ADBE', 'AMD', 'AMGN', 'AMZN', 'AVGO', 'AXP',
    'BA', 'BAC', 'BLK', 'BMY', 'BRK-B', 'C', 'CAT', 'CHTR', 'CL', 'CMCSA',
    'COF', 'COP', 'COST', 'CRM', 'CSCO', 'CVS', 'CVX', 'D', 'DE', 'DHR',
    'DIS', 'DOW', 'DUK', 'EMR', 'EXC', 'F', 'FDX', 'GD', 'GE', 'GILD',
    'GM', 'GOOG', 'GOOGL', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'INTU', 'ISRG',
    'JNJ', 'JPM', 'KHC', 'KO', 'LIN', 'LLY', 'LMT', 'LOW', 'MA', 'MCD',
    'MDLZ', 'MDT', 'MET', 'META', 'MMM', 'MO', 'MRK', 'MS', 'MSFT', 'NEE',
    'NFLX', 'NKE', 'NVDA', 'ORCL', 'PEP', 'PFE', 'PG', 'PM', 'PYPL', 'QCOM',
    'RTX', 'SBUX', 'SCHW', 'SO', 'SPG', 'T', 'TGT', 'TMO', 'TMUS', 'TXN',
    'UNH', 'UNP', 'UPS', 'USB', 'V', 'VZ', 'WBA', 'WFC', 'WMT', 'XOM'
]

try:
    sp500_df = pd.read_csv('https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv')
    all_tickers = sp500_df['Symbol'].tolist()
    print(f"[Universe] Found {len(all_tickers)} S&P 500 tickers from Wikipedia.")
except Exception as e:
    print(f"[Universe] ⚠️ Wikipedia fetch failed ({e}). Using fallback list.")
    all_tickers = sorted(FALLBACK_TICKERS)

if UNIVERSE_SIZE is not None and UNIVERSE_SIZE < len(all_tickers):
    random.seed(RANDOM_SEED)
    tickers = sorted(random.sample(all_tickers, UNIVERSE_SIZE))
    print(f"[Universe] Sampled {len(tickers)} tickers (seed={RANDOM_SEED}).")
else:
    tickers = all_tickers
    print(f"[Universe] Using full universe: {len(tickers)} tickers.")

print(f"[Universe] ✅ First 10 tickers: {tickers[:10]}")

## Section 6 — DefeatBeta API Client

In [ ]:
# ============================================================
# Section 6: DefeatBeta API Client (Python library — no API key needed)
# ============================================================
print("[API] Setting up DefeatBeta API client...")
print("[API] Using defeatbeta-api Python library (free, no API key required).")

def get_ticker_client(symbol):
    """Create a DefeatBeta Ticker client for a given symbol."""
    return Ticker(symbol.upper())

# Quick test
try:
    _test = get_ticker_client("AAPL")
    print("[API] ✅ DefeatBeta API client ready. Test: AAPL ticker created.")
except Exception as e:
    print(f"[API] ⚠️ Warning: {e}")
    print("[API] Will retry during data collection.")

## Section 7 — Fetch Earnings Call Transcripts

For each (ticker, quarter), we fetch available earnings call transcripts via DefeatBeta API and match them to calendar quarters by `report_date`.

In [ ]:
# ============================================================
# Section 7: Fetch Earnings Call Transcripts
# ============================================================
print("[Transcripts] Fetching earnings call transcripts...")
print(f"[Transcripts] Universe: {len(tickers)} tickers × {len(QUARTERS)} quarters")

TRANSCRIPTS_DIR = os.path.join(DRIVE_BASE_PATH, "data/raw/earnings_calls")
TRANSCRIPTS_CACHE = os.path.join(DRIVE_BASE_PATH, "data/processed/transcripts_cache.parquet")

# ---- Fast path: load single-file cache ----
if os.path.exists(TRANSCRIPTS_CACHE):
    print("[Transcripts] Loading from consolidated cache...")
    transcripts_df = pd.read_parquet(TRANSCRIPTS_CACHE)
    # Filter to current universe & quarters
    q_labels = [q['quarter'] for q in QUARTERS]
    transcripts_df = transcripts_df[
        transcripts_df['ticker'].isin(tickers) & transcripts_df['quarter'].isin(q_labels)
    ].reset_index(drop=True)
    total = len(transcripts_df)
    total_found = transcripts_df['has_transcript'].sum()
    total_missing = total - total_found
    print(f"[Transcripts] ✅ Loaded from cache. {total} records.")
    print(f"   Transcripts found    : {total_found}  ({100*total_found/max(total,1):.1f}%)")
    print(f"   Missing              : {total_missing}  ({100*total_missing/max(total,1):.1f}%)")
else:
    # ---- Slow path: fetch from API / individual file cache ----
    transcript_records = []
    total_found = 0
    total_missing = 0

    for ticker_sym in tqdm(tickers, desc="Fetching Transcripts"):
        cache_file = os.path.join(TRANSCRIPTS_DIR, f"{ticker_sym}_transcripts_list.json")

        if os.path.exists(cache_file):
            with open(cache_file, 'r') as f:
                transcript_list = json.load(f)
        else:
            try:
                tk = get_ticker_client(ticker_sym)
                tc_obj = tk.earning_call_transcripts()
                tc_list_df = tc_obj.get_transcripts_list()
                if tc_list_df is not None and len(tc_list_df) > 0:
                    transcript_list = tc_list_df.to_dict('records')
                else:
                    transcript_list = []
            except Exception as e:
                transcript_list = []
            with open(cache_file, 'w') as f:
                json.dump(transcript_list, f, default=str)

        for qinfo in QUARTERS:
            q_label = qinfo['quarter']
            q_start = qinfo['start_date']
            q_end = qinfo['end_date']
            search_start = q_start - timedelta(days=15)
            search_end = q_end + timedelta(days=60)

            transcript_cache = os.path.join(TRANSCRIPTS_DIR, f"{ticker_sym}_{q_label}.txt")

            if os.path.exists(transcript_cache):
                with open(transcript_cache, 'r') as f:
                    transcript_text = f.read()
            else:
                best_match = None
                best_date = None
                for t in transcript_list:
                    try:
                        rd = pd.to_datetime(t.get('report_date', '')).date()
                    except:
                        continue
                    if search_start <= rd <= search_end:
                        if best_date is None or rd > best_date:
                            best_match = t
                            best_date = rd

                transcript_text = ""
                if best_match is not None:
                    try:
                        tk = get_ticker_client(ticker_sym)
                        tc_obj = tk.earning_call_transcripts()
                        fy = int(best_match['fiscal_year'])
                        fq = int(best_match['fiscal_quarter'])
                        tc_df = tc_obj.get_transcript(fy, fq)
                        if tc_df is not None and len(tc_df) > 0:
                            parts = []
                            for _, row in tc_df.iterrows():
                                speaker = row.get('speaker', 'Unknown')
                                content = row.get('content', '')
                                parts.append(f"{speaker}: {content}")
                            transcript_text = "\n".join(parts)
                    except Exception as e:
                        pass
                with open(transcript_cache, 'w') as f:
                    f.write(transcript_text)

            has = len(transcript_text.strip()) > 0
            transcript_records.append({
                'ticker': ticker_sym, 'quarter': q_label,
                'transcript': transcript_text,
                'has_transcript': has
            })
            if has:
                total_found += 1
            else:
                total_missing += 1

    transcripts_df = pd.DataFrame(transcript_records)
    # Save consolidated cache
    transcripts_df.to_parquet(TRANSCRIPTS_CACHE, index=False)
    total = len(transcripts_df)
    print(f"[Transcripts] ✅ Done. Consolidated cache saved.")
    print(f"   Total stock-quarters : {total}")
    print(f"   Transcripts found    : {total_found}  ({100*total_found/max(total,1):.1f}%)")
    print(f"   Missing              : {total_missing}  ({100*total_missing/max(total,1):.1f}%)")

## Section 8 — Fetch News Data

In [ ]:
# ============================================================
# Section 8: Fetch News Data
# ============================================================
print("[News] Fetching news data...")
print(f"[News] Universe: {len(tickers)} tickers × {len(QUARTERS)} quarters")

NEWS_DIR = os.path.join(DRIVE_BASE_PATH, "data/raw/news")
NEWS_CACHE = os.path.join(DRIVE_BASE_PATH, "data/processed/news_cache.parquet")

# ---- Fast path: load consolidated cache (only if it has actual data) ----
_news_loaded = False
if os.path.exists(NEWS_CACHE):
    _tmp = pd.read_parquet(NEWS_CACHE)
    if _tmp['has_news'].sum() > 0:
        print("[News] Loading from consolidated cache...")
        q_labels = [q['quarter'] for q in QUARTERS]
        news_df_records = _tmp[
            _tmp['ticker'].isin(tickers) & _tmp['quarter'].isin(q_labels)
        ].reset_index(drop=True)
        # Restore articles from JSON string
        news_df_records['articles'] = news_df_records['articles_json'].apply(json.loads)
        total = len(news_df_records)
        total_with_news = int(news_df_records['has_news'].sum())
        print(f"[News] ✅ Loaded from cache. {total} records.")
        print(f"   Stock-quarters with ≥1 article : {total_with_news}  ({100*total_with_news/max(total,1):.1f}%)")
        print(f"   Stock-quarters with no news    : {total - total_with_news}  ({100*(total - total_with_news)/max(total,1):.1f}%)")
        _news_loaded = True
    else:
        print("[News] ⚠️ Existing cache has 0 articles — will re-fetch.")
    del _tmp

if not _news_loaded:
    os.makedirs(NEWS_DIR, exist_ok=True)
    news_records = []
    total_with_news = 0
    total_no_news = 0

    for ticker_sym in tqdm(tickers, desc="Fetching News"):
        # Get full news list for this ticker once
        try:
            tk = get_ticker_client(ticker_sym)
            news_list_df = tk.news().get_news_list()  # DataFrame with uuid, title, report_date, news, ...
            if news_list_df is not None and len(news_list_df) > 0:
                news_list_df['report_date'] = pd.to_datetime(news_list_df['report_date']).dt.date
            else:
                news_list_df = pd.DataFrame()
        except Exception as e:
            news_list_df = pd.DataFrame()

        for qinfo in QUARTERS:
            q_label = qinfo['quarter']
            q_start = qinfo['start_date']
            q_end   = qinfo['end_date'] + timedelta(days=45)

            articles = []
            if len(news_list_df) > 0:
                # Filter news within the quarter window
                mask = (news_list_df['report_date'] >= q_start) & (news_list_df['report_date'] <= q_end)
                q_news = news_list_df[mask].head(3)

                for _, row in q_news.iterrows():
                    # Extract content from the nested 'news' column (list of paragraphs)
                    content = ""
                    if isinstance(row.get('news'), list):
                        paragraphs = [p.get('paragraph_content', p.get('content', ''))
                                      for p in row['news'] if isinstance(p, dict)]
                        content = " ".join(paragraphs)

                    articles.append({
                        'title': str(row.get('title', '')),
                        'date': str(row.get('report_date', '')),
                        'content': content
                    })

            has_news = len(articles) > 0
            news_records.append({
                'ticker': ticker_sym,
                'quarter': q_label,
                'articles': articles,
                'has_news': has_news,
                'num_articles': len(articles)
            })
            if has_news:
                total_with_news += 1
            else:
                total_no_news += 1

    news_df_records = pd.DataFrame(news_records)

    # Save consolidated cache
    news_cache_df = news_df_records[['ticker', 'quarter', 'has_news', 'num_articles']].copy()
    news_cache_df['articles_json'] = news_df_records['articles'].apply(json.dumps)
    news_cache_df.to_parquet(NEWS_CACHE, index=False)

    total = total_with_news + total_no_news
    print(f"[News] ✅ Done. Consolidated cache saved.")
    print(f"   Stock-quarters with ≥1 article : {total_with_news}  ({100*total_with_news/max(total,1):.1f}%)")
    print(f"   Stock-quarters with no news    : {total_no_news}  ({100*total_no_news/max(total,1):.1f}%)")

## Section 9 — Fetch Price Data

In [ ]:
# ============================================================
# Section 9: Fetch Price Data
# ============================================================
print("[Prices] Fetching price data...")

PRICES_DIR = os.path.join(DRIVE_BASE_PATH, "data/raw/prices")

# Determine date range needed
earliest_start = min(q['start_date'] for q in QUARTERS) - timedelta(days=10)
latest_rebalance = max(q['rebalance_date'] for q in QUARTERS) + timedelta(days=10)

price_data = {}
api_success = 0
yf_used = 0
failed = 0

for ticker_sym in tqdm(tickers, desc="Fetching Prices"):
    cache_file = os.path.join(PRICES_DIR, f"{ticker_sym}_prices.parquet")

    if os.path.exists(cache_file):
        try:
            df = pd.read_parquet(cache_file)
            price_data[ticker_sym] = df
            api_success += 1
            continue
        except:
            pass

    # Try DefeatBeta API first
    df = None
    try:
        tk = get_ticker_client(ticker_sym)
        price_df = tk.price(
            start_date=str(earliest_start),
            end_date=str(latest_rebalance)
        )
        if price_df is not None and len(price_df) > 0:
            price_df = price_df.copy()
            # Normalize column names
            price_df.columns = [c.lower() for c in price_df.columns]
            if 'date' in price_df.columns:
                price_df['date'] = pd.to_datetime(price_df['date'])
                price_df = price_df.sort_values('date').reset_index(drop=True)
            df = price_df
            api_success += 1
    except Exception as e:
        pass

    # Fallback to yfinance
    if df is None or len(df) == 0:
        try:
            yf_ticker = yf.Ticker(ticker_sym)
            yf_df = yf_ticker.history(start=str(earliest_start), end=str(latest_rebalance))
            if len(yf_df) > 0:
                yf_df = yf_df.reset_index()
                yf_df.columns = [c.lower() for c in yf_df.columns]
                df = yf_df
                yf_used += 1
            else:
                failed += 1
        except Exception as e:
            failed += 1

    if df is not None and len(df) > 0:
        price_data[ticker_sym] = df
        try:
            df.to_parquet(cache_file, index=False)
        except:
            pass
    else:
        failed += 1

print(f"[Prices] ✅ Done.")
print(f"   API success     : {api_success}")
print(f"   yfinance used   : {yf_used}")
print(f"   Failed          : {failed}")
print(f"   Coverage        : {len(price_data)}/{len(tickers)} tickers")

## Section 10 — Build Quarterly Text Panel

In [ ]:
# ============================================================
# Section 10: Build Quarterly Text Panel
# ============================================================
print("[Panel] Building quarterly text panel...")

PANEL_PATH = os.path.join(DRIVE_BASE_PATH, "data/processed/quarterly_text_panel.parquet")

# Load tokenizer for truncation (CPU only)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_HF_ID, use_fast=True)
print("[Panel] Tokenizer loaded for truncation.")

# Create lookup dicts
transcript_lookup = {}
for _, row in transcripts_df.iterrows():
    transcript_lookup[(row['ticker'], row['quarter'])] = row['transcript']

news_lookup = {}
for _, row in news_df_records.iterrows():
    news_lookup[(row['ticker'], row['quarter'])] = row['articles']

panel_records = []
token_lengths = []

for ticker_sym in tqdm(tickers, desc="Building Panel"):
    for qinfo in QUARTERS:
        q_label = qinfo['quarter']

        # Build combined text
        parts = []
        parts.append(f"[Company Information]")
        parts.append(f"Ticker: {ticker_sym}")
        parts.append(f"Quarter: {q_label}")
        parts.append("")

        # Earnings call transcript
        transcript = transcript_lookup.get((ticker_sym, q_label), "")
        has_transcript = len(transcript.strip()) > 0
        parts.append("[Earnings Call Transcript]")
        if has_transcript:
            parts.append(transcript)
        else:
            parts.append("No transcript available for this quarter.")
        parts.append("")

        # News articles
        articles = news_lookup.get((ticker_sym, q_label), [])
        has_news = len(articles) > 0
        for i, article in enumerate(articles[:3], 1):
            parts.append(f"[News {i}]")
            parts.append(f"Title: {article.get('title', 'N/A')}")
            parts.append(f"Date: {article.get('date', 'N/A')}")
            parts.append(f"Content: {article.get('content', 'N/A')}")
            parts.append("")

        combined_text = "\n".join(parts)

        # Truncate if too long
        token_ids = tokenizer.encode(combined_text, add_special_tokens=False)
        if len(token_ids) > MAX_INPUT_TOKENS:
            token_ids = token_ids[:MAX_INPUT_TOKENS]
            combined_text = tokenizer.decode(token_ids, skip_special_tokens=True)

        token_len = min(len(token_ids), MAX_INPUT_TOKENS)
        token_lengths.append(token_len)

        panel_records.append({
            'ticker': ticker_sym,
            'quarter': q_label,
            'combined_text': combined_text,
            'has_transcript': has_transcript,
            'has_news': has_news,
            'token_length': token_len
        })

panel_df = pd.DataFrame(panel_records)
panel_df.to_parquet(PANEL_PATH, index=False)

n_with_transcript = panel_df['has_transcript'].sum()
n_with_news = panel_df['has_news'].sum()
total = len(panel_df)
print(f"[Panel] ✅ Done. Saved to {PANEL_PATH}")
print(f"   Total records            : {total}")
print(f"   Records with transcript  : {n_with_transcript}  ({100*n_with_transcript/total:.1f}%)")
print(f"   Records with ≥1 news     : {n_with_news}  ({100*n_with_news/total:.1f}%)")
print(f"   Avg token length         : {np.mean(token_lengths):.0f}  (max: {max(token_lengths)}, min: {min(token_lengths)})")

## Section 11 — Load DeepSeek Model + LoRA

Loading the base model with 4-bit quantization (~4–5 GB VRAM) and attaching the fine-tuned LoRA adapter from Google Drive.

In [ ]:
# ============================================================
# Section 11: Load DeepSeek Model + LoRA
# ============================================================
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Verify LoRA adapter files exist
adapter_config_path = os.path.join(LORA_ADAPTER_PATH, "adapter_config.json")
adapter_model_path  = os.path.join(LORA_ADAPTER_PATH, "adapter_model.safetensors")

assert os.path.exists(adapter_config_path), \
    f"❌ adapter_config.json not found at {adapter_config_path}"
assert os.path.exists(adapter_model_path), \
    f"❌ adapter_model.safetensors not found at {adapter_model_path}"
print("[Model] ✅ LoRA adapter files verified.")

# Step 1: Tokenizer (already loaded in Section 10, reuse it)
print("[Model] Step 1/3: Tokenizer already loaded. ✅")

# Step 2: Base model
print("[Model] Step 2/3: Loading base model weights...")
if LOAD_IN_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_HF_ID,
        quantization_config=quant_config,
        device_map="auto"
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_HF_ID,
        torch_dtype=torch.float16,
        device_map="auto"
    )
print("[Model] Base model loaded. ✅")

# Step 3: LoRA adapter
print("[Model] Step 3/3: Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)
model.eval()
print("[Model] LoRA adapter attached. ✅")

device = next(model.parameters()).device
print(f"[Model] ✅ Model ready. Device: {device}")
# if torch.cuda.is_available():
#     mem_used = torch.cuda.memory_allocated() / 1e9
#     mem_total = torch.cuda.get_device_properties(0).total_mem / 1e9
#     print(f"[Model] GPU memory: {mem_used:.1f} GB used / {mem_total:.1f} GB total")

## Section 12 — LLM Inference

This is the most time-intensive section. It scores every stock-quarter record using the fine-tuned model. Checkpoints are saved every `CHECKPOINT_EVERY` records to Google Drive for resumability.

In [ ]:
# ============================================================
# Section 12: LLM Inference
# ============================================================

INFERENCE_CHECKPOINT_FILE = os.path.join(DRIVE_BASE_PATH, "outputs/inference_checkpoint.json")
PREDICTIONS_PATH = os.path.join(DRIVE_BASE_PATH, "data/processed/llm_predictions.parquet")

# ---------- Prompt template (DeepSeek-R1-Distill chat format) ----------
SYSTEM_PROMPT = """You are a quantitative financial analyst. You will be given company information, an earnings call transcript, and recent news. Assess the company's forward-looking investment attractiveness for the next quarter.

You MUST respond with a valid JSON object using exactly these fields:
{
  "llm_score": <integer from 0 to 100>,
  "direction": "<positive|neutral|negative>",
  "confidence": "<low|medium|high>",
  "reasoning_summary": "<1-2 sentence explanation>"
}

Scoring guide:
- 0-20: Very negative outlook (deteriorating fundamentals, guidance cuts, major risks)
- 21-40: Negative outlook (below expectations, headwinds)
- 41-60: Neutral outlook (mixed signals, in-line results)
- 61-80: Positive outlook (beats expectations, tailwinds)
- 81-100: Very positive outlook (exceptional results, strong guidance raise)

Example output:
{"llm_score": 62, "direction": "positive", "confidence": "medium", "reasoning_summary": "Revenue beat expectations by 5% and management raised full-year guidance, but margin pressure from rising costs tempers the outlook."}

IMPORTANT: Output ONLY the JSON object. No other text before or after it."""

def build_prompt(combined_text):
    """Build prompt using DeepSeek-R1-Distill-Llama chat template."""
    return (
        f"<|begin▁of▁sentence|>"
        f"<|User|>\n"
        f"{SYSTEM_PROMPT}\n\n"
        f"--- BEGIN DATA ---\n"
        f"{combined_text}\n"
        f"--- END DATA ---\n\n"
        f"Now output your JSON assessment.\n"
        f"<|Assistant|>\n"
    )

# ---------- Robust JSON parser (handles <think> blocks) ----------
def parse_llm_output(raw_text):
    """Parse JSON from model output, handling <think>...</think> reasoning blocks."""

    text = raw_text

    # Strip everything inside <think>...</think> blocks (greedy)
    text_after_think = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

    # Also handle case where output starts with </think> (no opening tag)
    if '</think>' in text:
        text_after_think = text.split('</think>')[-1].strip()

    # Try to find JSON in the cleaned text first, then fall back to raw
    for candidate in [text_after_think, raw_text]:
        # Find all {...} blocks and try each
        matches = re.findall(r'\{[^{}]*\}', candidate, re.DOTALL)
        for match_str in matches:
            try:
                result = json.loads(match_str)
                if 'llm_score' in result:
                    score = int(result.get('llm_score', 50))
                    score = max(0, min(100, score))
                    direction = str(result.get('direction', 'neutral')).lower().strip()
                    if direction not in ('positive', 'neutral', 'negative'):
                        direction = 'neutral'
                    confidence = str(result.get('confidence', 'low')).lower().strip()
                    if confidence not in ('low', 'medium', 'high'):
                        confidence = 'low'
                    reasoning = str(result.get('reasoning_summary', ''))[:500]
                    return {
                        'llm_score': score,
                        'direction': direction,
                        'confidence': confidence,
                        'reasoning_summary': reasoning,
                        'parse_error': False
                    }
            except (json.JSONDecodeError, ValueError, TypeError):
                continue

    # Last resort: try to extract score from natural language in post-think text
    if text_after_think:
        score_match = re.search(r'(?:score|rating)[:\s]*(\d{1,3})', text_after_think, re.IGNORECASE)
        dir_match = re.search(r'\b(positive|negative|neutral)\b', text_after_think, re.IGNORECASE)
        if score_match:
            score = max(0, min(100, int(score_match.group(1))))
            direction = dir_match.group(1).lower() if dir_match else 'neutral'
            return {
                'llm_score': score,
                'direction': direction,
                'confidence': 'low',
                'reasoning_summary': text_after_think[:200].strip(),
                'parse_error': False
            }

    return {
        'llm_score': 50,
        'direction': 'neutral',
        'confidence': 'low',
        'reasoning_summary': 'parse_error',
        'parse_error': True
    }

# ---------- Load existing checkpoint ----------
predictions = {}
if os.path.exists(INFERENCE_CHECKPOINT_FILE):
    with open(INFERENCE_CHECKPOINT_FILE, 'r') as f:
        predictions = json.load(f)
    print(f"[Inference] Loaded checkpoint with {len(predictions)} existing records.")

# ---------- Determine remaining work ----------
all_keys = [(row['ticker'], row['quarter']) for _, row in panel_df.iterrows()]
remaining = [(t, q) for t, q in all_keys if f"{t}_{q}" not in predictions]

print(f"[Inference] Starting inference on {len(remaining)} remaining records")
print(f"[Inference] Total records: {len(all_keys)}  |  Already done: {len(all_keys) - len(remaining)}")
if len(remaining) > 0:
    est_time = len(remaining) * 5 / 60
    print(f"[Inference] Estimated time: ~{est_time:.0f} minutes (5s/record)")

parse_errors = 0
start_time = time.time()

pbar = tqdm(remaining, desc="LLM Inference", total=len(remaining))
for idx, (ticker_sym, q_label) in enumerate(pbar):
    key = f"{ticker_sym}_{q_label}"

    # Get combined text
    row = panel_df[(panel_df['ticker'] == ticker_sym) & (panel_df['quarter'] == q_label)].iloc[0]
    combined_text = row['combined_text']

    # Build prompt with correct chat template
    prompt = build_prompt(combined_text)

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_INPUT_TOKENS + 512).to(device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only new tokens
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Parse
    parsed = parse_llm_output(raw_output)
    parsed['raw_output'] = raw_output[:1000]
    predictions[key] = parsed

    if parsed['parse_error']:
        parse_errors += 1

    pbar.set_postfix({
        'ticker': ticker_sym,
        'quarter': q_label,
        'score': parsed['llm_score'],
    })

    # Checkpoint
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        with open(INFERENCE_CHECKPOINT_FILE, 'w') as f:
            json.dump(predictions, f)
        print(f"   💾 Checkpoint saved: {idx + 1}/{len(remaining)} records")

# Final save
with open(INFERENCE_CHECKPOINT_FILE, 'w') as f:
    json.dump(predictions, f)

elapsed = time.time() - start_time
total_processed = len(remaining)
print(f"[Inference] ✅ Complete.")
print(f"   Total records processed  : {total_processed}")
print(f"   JSON parse errors        : {parse_errors}  ({100*parse_errors/max(total_processed,1):.1f}%)")
print(f"   Elapsed time             : {elapsed/60:.1f} minutes")
if total_processed > 0:
    print(f"   Avg time per record      : {elapsed/total_processed:.1f}s")

# Build predictions DataFrame
pred_rows = []
for (ticker_sym, q_label) in all_keys:
    key = f"{ticker_sym}_{q_label}"
    if key in predictions:
        pred = predictions[key]
        pred_rows.append({
            'ticker': ticker_sym,
            'quarter': q_label,
            'llm_score': pred['llm_score'],
            'direction': pred['direction'],
            'confidence': pred['confidence'],
            'reasoning_summary': pred['reasoning_summary'],
            'parse_error': pred.get('parse_error', False)
        })

predictions_df = pd.DataFrame(pred_rows)
predictions_df.to_parquet(PREDICTIONS_PATH, index=False)
print(f"[Inference] Predictions saved to {PREDICTIONS_PATH}")

## Section 13 — Portfolio Construction

In [ ]:
# ============================================================
# Section 13: Portfolio Construction
# ============================================================
print("[Portfolio] Constructing long-short portfolios...")

PORTFOLIO_PATH = os.path.join(DRIVE_BASE_PATH, "data/processed/portfolio_weights.parquet")

portfolio_rows = []

for qinfo in QUARTERS:
    q_label = qinfo['quarter']
    rebalance_date = qinfo['rebalance_date']

    q_preds = predictions_df[predictions_df['quarter'] == q_label].copy()
    q_preds = q_preds.sort_values('llm_score', ascending=False).reset_index(drop=True)
    n = len(q_preds)

    if n == 0:
        print(f"[Portfolio] {q_label} | ⚠️ No predictions, skipping.")
        continue

    n_long  = max(1, math.ceil(n * LONG_PCT))
    n_short = max(1, math.ceil(n * SHORT_PCT))

    long_stocks  = q_preds.head(n_long)
    short_stocks = q_preds.tail(n_short)

    for _, row in long_stocks.iterrows():
        portfolio_rows.append({
            'ticker': row['ticker'], 'quarter': q_label,
            'rebalance_date': rebalance_date, 'side': 'long',
            'weight': 1.0 / n_long, 'llm_score': row['llm_score']
        })

    for _, row in short_stocks.iterrows():
        portfolio_rows.append({
            'ticker': row['ticker'], 'quarter': q_label,
            'rebalance_date': rebalance_date, 'side': 'short',
            'weight': 1.0 / n_short, 'llm_score': row['llm_score']
        })

    long_scores = long_stocks['llm_score'].values
    short_scores = short_stocks['llm_score'].values
    print(f"[Portfolio] {q_label} | n={n} | Long: {n_long} (score {long_scores.min():.0f}–{long_scores.max():.0f}) | Short: {n_short} (score {short_scores.min():.0f}–{short_scores.max():.0f})")

portfolio_df = pd.DataFrame(portfolio_rows)
portfolio_df.to_parquet(PORTFOLIO_PATH, index=False)
print(f"[Portfolio] ✅ Done. Saved to {PORTFOLIO_PATH}")

## Section 14 — Backtest

In [ ]:
# ============================================================
# Section 14: Backtest
# ============================================================
print("[Backtest] Running backtest...")

BACKTEST_PATH = os.path.join(DRIVE_BASE_PATH, "outputs/backtest_results.csv")

def get_price_on_date(ticker_sym, target_date, tolerance_days=10):
    """Get adjusted close price nearest to target_date within tolerance."""
    if ticker_sym not in price_data:
        return None
    df = price_data[ticker_sym].copy()

    # Try to find 'close' or 'adj close' or 'adjclose' column
    close_col = None
    for col in ['adj close', 'adjclose', 'adj_close', 'close']:
        if col in df.columns:
            close_col = col
            break
    if close_col is None:
        return None

    # Find date column
    date_col = None
    for col in ['date', 'Date']:
        if col in df.columns:
            date_col = col
            break
    if date_col is None:
        return None

    df[date_col] = pd.to_datetime(df[date_col]).dt.date
    target = target_date if isinstance(target_date, date) else target_date.date()

    # Find closest date within tolerance
    df['diff'] = df[date_col].apply(lambda d: abs((d - target).days))
    df_close = df[df['diff'] <= tolerance_days]
    if len(df_close) == 0:
        return None
    closest = df_close.loc[df_close['diff'].idxmin()]
    return float(closest[close_col])

backtest_results = []

for i in range(len(QUARTERS) - 1):
    q_current = QUARTERS[i]
    q_next = QUARTERS[i + 1]
    q_label = q_current['quarter']
    reb_date_t0 = q_current['rebalance_date']
    reb_date_t1 = q_next['rebalance_date']

    q_portfolio = portfolio_df[portfolio_df['quarter'] == q_label]
    longs  = q_portfolio[q_portfolio['side'] == 'long']
    shorts = q_portfolio[q_portfolio['side'] == 'short']

    # Long returns
    long_returns = []
    for _, row in longs.iterrows():
        p0 = get_price_on_date(row['ticker'], reb_date_t0)
        p1 = get_price_on_date(row['ticker'], reb_date_t1)
        if p0 is not None and p1 is not None and p0 > 0:
            ret = (p1 / p0) - 1.0
            long_returns.append(ret)

    # Short returns
    short_returns = []
    for _, row in shorts.iterrows():
        p0 = get_price_on_date(row['ticker'], reb_date_t0)
        p1 = get_price_on_date(row['ticker'], reb_date_t1)
        if p0 is not None and p1 is not None and p0 > 0:
            ret = (p1 / p0) - 1.0
            short_returns.append(ret)

    long_ret  = np.mean(long_returns) if long_returns else 0.0
    short_ret = np.mean(short_returns) if short_returns else 0.0
    ls_ret = long_ret - short_ret

    backtest_results.append({
        'quarter': q_label,
        'next_quarter': q_next['quarter'],
        'rebalance_date_t0': str(reb_date_t0),
        'rebalance_date_t1': str(reb_date_t1),
        'long_return': long_ret,
        'short_return': short_ret,
        'ls_return': ls_ret,
        'n_long': len(long_returns),
        'n_short': len(short_returns)
    })

    print(f"[Backtest] {q_label} → {q_next['quarter']} | Long: {long_ret:+.2%} | Short: {short_ret:+.2%} | L-S: {ls_ret:+.2%}  ({len(long_returns)}L/{len(short_returns)}S)")

backtest_df = pd.DataFrame(backtest_results)

# Cumulative returns
backtest_df['cum_ls'] = (1 + backtest_df['ls_return']).cumprod()
backtest_df['cum_long'] = (1 + backtest_df['long_return']).cumprod()
backtest_df['cum_short'] = (1 + backtest_df['short_return']).cumprod()

backtest_df.to_csv(BACKTEST_PATH, index=False)
print(f"[Backtest] ✅ Done. Results saved to {BACKTEST_PATH}")

## Section 15 — Performance Metrics

In [ ]:
# ============================================================
# Section 15: Performance Metrics
# ============================================================
print("[Metrics] Computing performance metrics...")

METRICS_PATH = os.path.join(DRIVE_BASE_PATH, "outputs/performance_summary.json")

ls_returns = backtest_df['ls_return'].values
long_returns_all = backtest_df['long_return'].values
short_returns_all = backtest_df['short_return'].values
n_periods = len(ls_returns)

# Cumulative return
cum_return = np.prod(1 + ls_returns) - 1

# Annualized return (quarterly → annual)
ann_return = (1 + cum_return) ** (4.0 / n_periods) - 1 if n_periods > 0 else 0

# Annualized volatility
ann_vol = np.std(ls_returns, ddof=1) * np.sqrt(4) if n_periods > 1 else 0

# Sharpe ratio (assuming 0 risk-free rate for simplicity)
sharpe = ann_return / ann_vol if ann_vol > 0 else 0

# Max drawdown
cum_equity = np.cumprod(1 + ls_returns)
running_max = np.maximum.accumulate(cum_equity)
drawdowns = cum_equity / running_max - 1
max_drawdown = np.min(drawdowns) if len(drawdowns) > 0 else 0

# Hit rates
long_hit = np.mean(long_returns_all > 0) if len(long_returns_all) > 0 else 0
short_hit = np.mean(short_returns_all < 0) if len(short_returns_all) > 0 else 0

metrics = {
    'cumulative_return': round(cum_return, 4),
    'annualized_return': round(ann_return, 4),
    'annualized_volatility': round(ann_vol, 4),
    'sharpe_ratio': round(sharpe, 4),
    'max_drawdown': round(max_drawdown, 4),
    'long_hit_rate': round(long_hit, 4),
    'short_hit_rate': round(short_hit, 4),
    'num_quarters': int(n_periods)
}

with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"[Metrics] ✅ Performance Summary:")
print(f"   {'Metric':<25} {'Value':>12}")
print(f"   {'-'*37}")
print(f"   {'Cumulative Return':<25} {cum_return:>11.2%}")
print(f"   {'Annualized Return':<25} {ann_return:>11.2%}")
print(f"   {'Annualized Volatility':<25} {ann_vol:>11.2%}")
print(f"   {'Sharpe Ratio':<25} {sharpe:>12.2f}")
print(f"   {'Max Drawdown':<25} {max_drawdown:>11.2%}")
print(f"   {'Long Hit Rate':<25} {long_hit:>11.2%}")
print(f"   {'Short Hit Rate':<25} {short_hit:>11.2%}")
print(f"   {'Quarters':<25} {n_periods:>12d}")
print(f"\n   Saved to {METRICS_PATH}")

## Section 16 — Visualization

In [ ]:
# ============================================================
# Section 16: Visualization
# ============================================================
print("[Viz] Generating performance plots...")

PLOT_PATH = os.path.join(DRIVE_BASE_PATH, "outputs/performance_plot.png")

fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [2, 1]})

# --- Subplot 1: Cumulative Return Curves ---
ax1 = axes[0]
quarters_labels = backtest_df['quarter'].values
x = range(len(quarters_labels))

ax1.plot(x, backtest_df['cum_long'].values, marker='o', label='Long-Only', color='#2196F3', linewidth=2)
ax1.plot(x, backtest_df['cum_short'].values, marker='s', label='Short-Only', color='#FF5722', linewidth=2)
ax1.plot(x, backtest_df['cum_ls'].values, marker='^', label='Long-Short', color='#4CAF50', linewidth=2.5)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xticks(x)
ax1.set_xticklabels(quarters_labels, rotation=45, ha='right')
ax1.set_ylabel('Cumulative Return (Growth of $1)')
ax1.set_title('LLM Long-Short Equity Strategy — Cumulative Returns', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# --- Subplot 2: Quarterly L-S Bar Chart ---
ax2 = axes[1]
colors = ['#4CAF50' if r >= 0 else '#F44336' for r in backtest_df['ls_return'].values]
ax2.bar(x, backtest_df['ls_return'].values * 100, color=colors, alpha=0.8, edgecolor='white')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(quarters_labels, rotation=45, ha='right')
ax2.set_ylabel('L-S Return (%)')
ax2.set_title('Quarterly Long-Short Returns', fontsize=12)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f"[Viz] ✅ Plot saved to {PLOT_PATH}")

## Section 17 — Save Reasoning Cases

In [ ]:
# ============================================================
# Section 17: Save Reasoning Cases
# ============================================================
print("[Reasoning] Extracting top reasoning cases per quarter...")

REASONING_PATH = os.path.join(DRIVE_BASE_PATH, "outputs/reasoning_cases.json")

reasoning_cases = []

for qinfo in QUARTERS:
    q_label = qinfo['quarter']
    q_preds = predictions_df[predictions_df['quarter'] == q_label].copy()
    q_preds = q_preds.sort_values('llm_score', ascending=False)

    top_long = q_preds.head(3)[['ticker', 'llm_score', 'reasoning_summary']].to_dict('records')
    top_short = q_preds.tail(3)[['ticker', 'llm_score', 'reasoning_summary']].to_dict('records')

    reasoning_cases.append({
        'quarter': q_label,
        'long_top3': top_long,
        'short_top3': top_short
    })

with open(REASONING_PATH, 'w') as f:
    json.dump(reasoning_cases, f, indent=2, default=str)

print(f"[Reasoning] ✅ Saved {len(reasoning_cases)} quarter reasoning cases to {REASONING_PATH}")

# Display a sample
print(f"\n--- Sample: {reasoning_cases[0]['quarter']} ---")
print("Top 3 LONG:")
for r in reasoning_cases[0]['long_top3']:
    print(f"  {r['ticker']} (score={r['llm_score']}): {r['reasoning_summary'][:100]}")
print("Top 3 SHORT:")
for r in reasoning_cases[0]['short_top3']:
    print(f"  {r['ticker']} (score={r['llm_score']}): {r['reasoning_summary'][:100]}")

## Section 18 — Final Summary

In [ ]:
# ============================================================
# Section 18: Final Summary
# ============================================================
print("=" * 60)
print("  LLM LONG-SHORT EQUITY STRATEGY — FINAL SUMMARY")
print("=" * 60)
print()
print(f"  Universe         : {len(tickers)} stocks")
print(f"  Quarters         : {len(QUARTERS)} ({QUARTERS[0]['quarter']} to {QUARTERS[-1]['quarter']})")
print(f"  Model            : {BASE_MODEL_HF_ID}")
print(f"  LoRA adapter     : {LORA_ADAPTER_PATH}")
print(f"  Quantization     : {'4-bit NF4' if LOAD_IN_4BIT else 'fp16'}")
print()
print(f"  --- Data Coverage ---")
print(f"  Transcripts      : {transcripts_df['has_transcript'].sum()}/{len(transcripts_df)} ({100*transcripts_df['has_transcript'].mean():.1f}%)")
print(f"  News articles    : {news_df_records['has_news'].sum()}/{len(news_df_records)} ({100*news_df_records['has_news'].mean():.1f}%)")
print(f"  Price data       : {len(price_data)}/{len(tickers)} tickers")
print()
print(f"  --- Strategy Performance ---")
print(f"  Cumulative Return    : {metrics['cumulative_return']:.2%}")
print(f"  Annualized Return    : {metrics['annualized_return']:.2%}")
print(f"  Sharpe Ratio         : {metrics['sharpe_ratio']:.2f}")
print(f"  Max Drawdown         : {metrics['max_drawdown']:.2%}")
print(f"  Long Hit Rate        : {metrics['long_hit_rate']:.2%}")
print(f"  Short Hit Rate       : {metrics['short_hit_rate']:.2%}")
print()
print(f"  --- Output Files ---")

output_files = [
    ('quarterly_text_panel.parquet', 'data/processed'),
    ('llm_predictions.parquet', 'data/processed'),
    ('portfolio_weights.parquet', 'data/processed'),
    ('backtest_results.csv', 'outputs'),
    ('performance_summary.json', 'outputs'),
    ('reasoning_cases.json', 'outputs'),
    ('performance_plot.png', 'outputs'),
    ('inference_checkpoint.json', 'outputs'),
]
for fname, subdir in output_files:
    fpath = os.path.join(DRIVE_BASE_PATH, subdir, fname)
    exists = "✅" if os.path.exists(fpath) else "❌"
    print(f"  {exists} {subdir}/{fname}")

print()
print("=" * 60)
print("  Pipeline complete\! All results saved to Google Drive.")
print("=" * 60)